# Assignment #1: Basic Notions of Probability
**Questions answered: Q1, Q2, Q8, Q9, Q10**

# Q1) The Probability of Shooting at a Target

A point is chosen at random from a disk of radius $10$. Let $A$ be the event that the point lies within $1$ unit of the boundary.

1. Model the experiment assuming the point is uniformly distributed with respect to area, and compute $P(A)$.
2. Model the experiment assuming the distance to the point from the center is chosen uniformly from $[0,10]$, and the direction is chosen independently and uniformly from $[0,2\pi)$ and compute $P(A)$.
3. Explain why the two answers differ.

## Answer to Q1

### Part 1: Uniform distribution with respect to area

**Probability space:** Let $\Omega$ be the closed disk of radius 10 centered at the origin,
$$\Omega = \{(x,y)\in\mathbb{R}^2 : x^2+y^2 \le 100\}.$$
Let $\mathfrak{F} = \mathfrak{B}(\Omega)$ (the Borel $\sigma$-algebra restricted to $\Omega$), and let $P$ be the probability measure proportional to 2D Lebesgue measure:
$$P(B) = \frac{\text{Area}(B)}{\text{Area}(\Omega)} = \frac{\text{Area}(B)}{100\pi}.$$

**Event $A$:** The point lies within 1 unit of the boundary, i.e., its distance from the center satisfies $9 \le r \le 10$. This is an annular region.

**Computation:**
$$P(A) = \frac{\text{Area of annulus}}{\text{Area of disk}} = \frac{\pi(10)^2 - \pi(9)^2}{\pi(10)^2} = \frac{100\pi - 81\pi}{100\pi} = \frac{19\pi}{100\pi} = \boxed{\frac{19}{100} = 0.19}.$$

---

### Part 2: Uniform distribution over radius and angle

**Probability space:** Here we use polar coordinates. Let
$$\Omega = [0,10] \times [0,2\pi),$$
with $\mathfrak{F} = \mathfrak{B}(\Omega)$ and $P$ the uniform product measure:
$$P(B) = \frac{\text{Lebesgue measure of } B}{10 \cdot 2\pi}.$$
The radius $R$ is uniform on $[0,10]$ and the angle $\Theta$ is uniform on $[0,2\pi)$, independently.

**Event $A$:** The point lies within 1 unit of the boundary, i.e., $R \ge 9$.

**Computation:** Since $R \sim \text{Uniform}[0,10]$,
$$P(A) = P(R \ge 9) = \frac{10 - 9}{10 - 0} = \boxed{\frac{1}{10} = 0.10}.$$

---

### Part 3: Why the two answers differ

The two models assign different probability measures to the same geometric space, so they represent fundamentally different notions of "random."

- **Model 1** (uniform over area) weights each region proportionally to its **area**. Points near the boundary of a disk occupy a disproportionately large area compared to points near the center (the annulus $9 \le r \le 10$ has area $19\pi$, which is nearly a fifth of the total area $100\pi$). This reflects the physical intuition of throwing a dart uniformly at a board.

- **Model 2** (uniform radius) weights radial distances equally. It assigns probability $1/10$ to each unit interval of radius, ignoring the fact that a thin annulus at radius $r$ has circumference $2\pi r$ — the area element $r\,dr\,d\theta$ is *not* accounted for. This model under-represents the outer region geometrically.

In other words, uniform over area uses the **area measure** $dA = r\,dr\,d\theta$, while Model 2 uses the **product measure** $dr\,d\theta$ (dropping the Jacobian factor $r$). The Jacobian factor $r$ is what makes the outer annulus have larger area, and its omission in Model 2 leads to a smaller probability for the event $A$.

This is precisely the content of the **change-of-variables theorem**: different parameterizations of the same space lead to different probability distributions unless the Jacobian is correctly incorporated.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Numerical verification via Monte Carlo
np.random.seed(42)
N = 1_000_000

# Model 1: uniform over area (rejection sampling)
x = np.random.uniform(-10, 10, N)
y = np.random.uniform(-10, 10, N)
in_disk = x**2 + y**2 <= 100
near_boundary = (x**2 + y**2 >= 81) & in_disk
p_model1 = near_boundary.sum() / in_disk.sum()
print(f"Model 1 (uniform area) Monte Carlo: P(A) ≈ {p_model1:.4f}  (exact: 0.19)")

# Model 2: uniform radius in [0,10], uniform angle
r = np.random.uniform(0, 10, N)
p_model2 = (r >= 9).mean()
print(f"Model 2 (uniform radius) Monte Carlo: P(A) ≈ {p_model2:.4f}  (exact: 0.10)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

theta = np.linspace(0, 2*np.pi, 300)
for ax, title, color_cond, N_pts in zip(axes,
    ['Model 1: Uniform Area', 'Model 2: Uniform Radius'],
    [None, None], [5000, 5000]):

    if ax == axes[0]:
        xs = np.random.uniform(-10, 10, N_pts*4)
        ys = np.random.uniform(-10, 10, N_pts*4)
        mask = xs**2 + ys**2 <= 100
        xs, ys = xs[mask][:N_pts], ys[mask][:N_pts]
        colors = np.where(xs**2 + ys**2 >= 81, 'red', 'steelblue')
    else:
        rs = np.random.uniform(0, 10, N_pts)
        ts = np.random.uniform(0, 2*np.pi, N_pts)
        xs, ys = rs*np.cos(ts), rs*np.sin(ts)
        colors = np.where(rs >= 9, 'red', 'steelblue')

    ax.scatter(xs, ys, c=colors, s=1, alpha=0.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    boundary_circle = plt.Circle((0, 0), 10, fill=False, color='black', lw=1.5)
    inner_circle = plt.Circle((0, 0), 9, fill=False, color='gray', lw=1, linestyle='--')
    ax.add_patch(boundary_circle)
    ax.add_patch(inner_circle)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='red', label='Within 1 of boundary'),
                       Patch(color='steelblue', label='Interior')], loc='upper right', fontsize=9)

plt.suptitle('Q1: Two Different Models of "Random Point in Disk"', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Red points = event A (within 1 unit of boundary). Note the different density patterns!")

# Q2) Bertrand's Paradox

A chord of a circle is chosen "at random." What is the probability that its length exceeds the side length of the inscribed equilateral triangle?

Show that different mathematically natural models of "random chord" lead to different answers.

## Answer to Q2

**Setup.** Without loss of generality, consider a circle of radius $R = 1$. The side length of an inscribed equilateral triangle is $\ell = R\sqrt{3} = \sqrt{3}$.

We present three classical models, each giving a different answer.

---

### Method 1: Random Endpoints (Uniform over pairs of angles)

**Model.** Choose two points independently and uniformly on the circle. By rotational symmetry, fix one endpoint at angle $0$; the other endpoint is uniform on $[0, 2\pi)$.

**Chord length.** If the second endpoint is at angle $\theta \in [0, 2\pi)$, the chord length is $2\sin(\theta/2)$.

**Condition for exceeding $\sqrt{3}$:**
$$2\sin(\theta/2) > \sqrt{3} \iff \sin(\theta/2) > \frac{\sqrt{3}}{2} \iff \frac{\theta}{2} \in \left(\frac{\pi}{3}, \frac{2\pi}{3}\right) \iff \theta \in \left(\frac{2\pi}{3}, \frac{4\pi}{3}\right).$$

**Probability:**
$$P = \frac{4\pi/3 - 2\pi/3}{2\pi} = \frac{2\pi/3}{2\pi} = \boxed{\frac{1}{3}}.$$

---

### Method 2: Random Midpoint (Uniform over the disk)

**Model.** A chord is uniquely determined by its midpoint (except for the degenerate case). Choose the midpoint uniformly over the disk.

**Chord length from midpoint.** If the midpoint is at distance $d$ from the center, the chord length is $2\sqrt{1 - d^2}$.

**Condition for exceeding $\sqrt{3}$:**
$$2\sqrt{1-d^2} > \sqrt{3} \iff 1 - d^2 > \frac{3}{4} \iff d^2 < \frac{1}{4} \iff d < \frac{1}{2}.$$

**Probability** (the midpoint must lie inside the disk of radius $1/2$):
$$P = \frac{\pi(1/2)^2}{\pi(1)^2} = \boxed{\frac{1}{4}}.$$

---

### Method 3: Random Radius (Fix a direction, uniform position on a diameter)

**Model.** Fix a direction (by symmetry, say vertical). A chord perpendicular to this direction is determined by where it crosses the diameter. Choose this crossing point uniformly on $[-1, 1]$.

**Chord length.** If the chord crosses the diameter at signed distance $d$ from the center, its length is $2\sqrt{1-d^2}$.

**Condition for exceeding $\sqrt{3}$:**
$$2\sqrt{1-d^2} > \sqrt{3} \iff d^2 < \frac{1}{4} \iff d \in \left(-\frac{1}{2}, \frac{1}{2}\right).$$

**Probability** (uniform on $[-1,1]$):
$$P = \frac{1/2 - (-1/2)}{1-(-1)} = \frac{1}{2} \cdot \frac{1}{1} = \boxed{\frac{1}{2}}.$$

---

### Summary and Discussion

| Method | Model | $P(\text{chord} > \sqrt{3})$ |
|--------|-------|---|
| Random Endpoints | Uniform on circle $\times$ circle | $1/3$ |
| Random Midpoint | Uniform over disk | $1/4$ |
| Random Radius | Uniform on diameter | $1/2$ |

**Why the answers differ:** The phrase "chosen at random" is ambiguous — it does not specify which probability measure is used on the space of all chords. Each method defines a different, mathematically legitimate probability measure on the same set of objects (chords of the circle), and each gives a different answer.

This is Bertrand's Paradox (1889): the probability of an event can depend crucially on the **choice of probability space** and **parameterization** of the sample space. There is no single "natural" or "correct" uniform distribution on chords, because the set of chords is not a linear space and does not carry a canonical invariant measure.

Jaynes (1983) argued that requiring **geometric invariance** (invariance under translation, rotation, and scaling) uniquely selects Method 2 (random midpoint, giving $P = 1/4$), but this remains a matter of philosophical debate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
N = 200_000
sqrt3 = np.sqrt(3)

# Method 1: Random endpoints
theta1 = np.random.uniform(0, 2*np.pi, N)
theta2 = np.random.uniform(0, 2*np.pi, N)
x1, y1 = np.cos(theta1), np.sin(theta1)
x2, y2 = np.cos(theta2), np.sin(theta2)
chord_len1 = np.sqrt((x2-x1)**2 + (y2-y1)**2)
p1 = (chord_len1 > sqrt3).mean()

# Method 2: Random midpoint in disk
r2 = np.sqrt(np.random.uniform(0, 1, N))  # uniform area
phi2 = np.random.uniform(0, 2*np.pi, N)
d2 = r2
chord_len2 = 2*np.sqrt(np.maximum(1 - d2**2, 0))
p2 = (chord_len2 > sqrt3).mean()

# Method 3: Random position on diameter
d3 = np.random.uniform(-1, 1, N)
chord_len3 = 2*np.sqrt(np.maximum(1 - d3**2, 0))
p3 = (chord_len3 > sqrt3).mean()

print("Bertrand's Paradox - Monte Carlo Verification")
print(f"Method 1 (Random Endpoints): P = {p1:.4f}  (exact: 1/3 ≈ {1/3:.4f})")
print(f"Method 2 (Random Midpoint):  P = {p2:.4f}  (exact: 1/4 = {1/4:.4f})")
print(f"Method 3 (Random Radius):    P = {p3:.4f}  (exact: 1/2 = {1/2:.4f})")

# Visualization: show chord distributions for each method
n_draw = 300
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = ['Method 1: Random Endpoints\nP = 1/3',
          'Method 2: Random Midpoint\nP = 1/4',
          'Method 3: Random Radius\nP = 1/2']

theta_c = np.linspace(0, 2*np.pi, 300)

for ax, title in zip(axes, titles):
    ax.plot(np.cos(theta_c), np.sin(theta_c), 'k-', lw=1.5)
    ax.set_aspect('equal'); ax.set_title(title, fontsize=11)
    ax.axis('off')

# Draw sample chords for each method
for i in range(n_draw):
    # Method 1
    a1, b1 = np.random.uniform(0,2*np.pi), np.random.uniform(0,2*np.pi)
    clen = np.sqrt((np.cos(b1)-np.cos(a1))**2 + (np.sin(b1)-np.sin(a1))**2)
    col = 'red' if clen > sqrt3 else 'lightblue'
    axes[0].plot([np.cos(a1),np.cos(b1)],[np.sin(a1),np.sin(b1)],color=col,alpha=0.3,lw=0.8)

    # Method 2
    rm = np.sqrt(np.random.uniform(0,1))
    pm = np.random.uniform(0,2*np.pi)
    mx, my = rm*np.cos(pm), rm*np.sin(pm)
    half = np.sqrt(max(1-rm**2,0))
    perp = pm + np.pi/2
    clen2 = 2*half
    col2 = 'red' if clen2 > sqrt3 else 'lightblue'
    axes[1].plot([mx-half*np.cos(perp), mx+half*np.cos(perp)],
                 [my-half*np.sin(perp), my+half*np.sin(perp)],color=col2,alpha=0.3,lw=0.8)

    # Method 3
    d = np.random.uniform(-1, 1)
    half3 = np.sqrt(max(1-d**2,0))
    clen3 = 2*half3
    col3 = 'red' if clen3 > sqrt3 else 'lightblue'
    axes[2].plot([-half3, half3],[d, d],color=col3,alpha=0.3,lw=0.8)

for ax in axes:
    ax.plot(np.cos(theta_c), np.sin(theta_c), 'k-', lw=1.5)

from matplotlib.patches import Patch
fig.legend(handles=[Patch(color='red', label=f'Length > √3'),
                    Patch(color='lightblue', label=f'Length ≤ √3')],
           loc='lower center', ncol=2, fontsize=10)
plt.suptitle("Bertrand's Paradox: Three Models of 'Random Chord'", fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0.07,1,1])
plt.show()

# Q8) Probability of Being a Girl?

A family has two children. Assume each child is equally likely to be a boy or a girl, independently of the other.

What is the probability that both children are girls, given that **at least one** of them is a girl?

## Answer to Q8

### Probability Space

Let $\Omega = \{BB,\, BG,\, GB,\, GG\}$, where the first letter denotes the older child and the second the younger child. Since each child is independently and equally likely to be a boy or girl:
$$P(\{BB\}) = P(\{BG\}) = P(\{GB\}) = P(\{GG\}) = \frac{1}{4}.$$

Let $\mathfrak{F} = \mathfrak{P}(\Omega)$ (the power set, all $2^4 = 16$ subsets).

### Events

- Let $A$ = "both children are girls" $= \{GG\}$.
- Let $B$ = "at least one child is a girl" $= \{BG, GB, GG\}$.

### Computation

We want $P(A \mid B)$. By Bayes' theorem (the definition of conditional probability):
$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}.$$

Since $A = \{GG\} \subseteq B = \{BG, GB, GG\}$, we have $A \cap B = A = \{GG\}$. Therefore:
$$P(A \cap B) = P(\{GG\}) = \frac{1}{4},$$
$$P(B) = P(\{BG\}) + P(\{GB\}) + P(\{GG\}) = \frac{1}{4} + \frac{1}{4} + \frac{1}{4} = \frac{3}{4}.$$

Thus:
$$\boxed{P(\text{both girls} \mid \text{at least one girl}) = \frac{1/4}{3/4} = \frac{1}{3}.}$$

### Intuition

Knowing that at least one child is a girl eliminates the outcome $BB$, leaving three equally likely outcomes: $BG$, $GB$, and $GG$. Of these three, only one ($GG$) corresponds to both children being girls, giving probability $\frac{1}{3}$.

> **Note:** This answer ($1/3$) is different from the seemingly similar question "given that the *older* child is a girl, what is the probability both are girls?" — that gives $1/2$. The distinction lies in *what specific information* the conditioning provides.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Enumerate the sample space
outcomes = ['BB', 'BG', 'GB', 'GG']
probs = {o: 0.25 for o in outcomes}

at_least_one_girl = [o for o in outcomes if 'G' in o]
both_girls = [o for o in outcomes if o == 'GG']

P_B = sum(probs[o] for o in at_least_one_girl)
P_A_and_B = sum(probs[o] for o in both_girls)
P_A_given_B = P_A_and_B / P_B

print("Sample Space:", outcomes)
print(f"\nEvent B (at least one girl): {at_least_one_girl}")
print(f"P(B) = {P_B}")
print(f"\nEvent A∩B (both girls): {both_girls}")
print(f"P(A∩B) = {P_A_and_B}")
print(f"\nP(both girls | at least one girl) = {P_A_and_B}/{P_B} = {P_A_given_B:.4f} ≈ 1/3")

# Visualization: probability table
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

colors = []
for o in outcomes:
    if o == 'BB':
        colors.append('#f5c6c6')  # eliminated
    elif o == 'GG':
        colors.append('#90EE90')  # event A
    else:
        colors.append('#ADD8E6')  # in B, not A

for i, (outcome, color) in enumerate(zip(outcomes, colors)):
    rect = mpatches.FancyBboxPatch((i*2, 0.5), 1.6, 2,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(rect)
    ax.text(i*2 + 0.8, 1.8, outcome, ha='center', va='center', fontsize=20, fontweight='bold')
    ax.text(i*2 + 0.8, 1.1, 'P = 1/4', ha='center', va='center', fontsize=11)
    if outcome == 'BB':
        ax.text(i*2 + 0.8, 0.2, '✗ Eliminated', ha='center', va='center', fontsize=9, color='red')
    elif outcome == 'GG':
        ax.text(i*2 + 0.8, 0.2, '✓ Event A', ha='center', va='center', fontsize=9, color='green')
    else:
        ax.text(i*2 + 0.8, 0.2, 'In B', ha='center', va='center', fontsize=9, color='steelblue')

ax.set_xlim(-0.3, 8.3); ax.set_ylim(-0.3, 3.5)
ax.set_title('Q8: P(both girls | at least one girl) = 1/3\n'
             'Conditioning on B={BG,GB,GG} leaves 3 equally likely outcomes; only GG satisfies A',
             fontsize=11, fontweight='bold')
legend_elements = [mpatches.Patch(facecolor='#f5c6c6', label='BB: eliminated by conditioning'),
                   mpatches.Patch(facecolor='#ADD8E6', label='In B, not A'),
                   mpatches.Patch(facecolor='#90EE90', label='GG: event A (both girls)')]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

# Q9) Information Available in a Discrete Random Variable

A factory inspects a product and records its quality using two different coding systems.

Each product falls into exactly one of the following four categories: $G_1$, $G_2$, $D_1$, $D_2$, with probabilities
$$P(\{G_1\})=0.50,\quad P(\{G_2\})=0.20,\quad P(\{D_1\})=0.10,\quad P(\{D_2\})=0.20.$$

Random variables: $X$ (good=0, defective=1) and $Y$ (full label: $G_1\to1, G_2\to2, D_1\to3, D_2\to4$).

## Answer to Q9

### Part 1: Probability Space

$$\Omega = \{G_1, G_2, D_1, D_2\}, \quad \mathfrak{F} = \mathfrak{P}(\Omega) \text{ (all 16 subsets)},$$
$$P(\{G_1\})=0.50,\; P(\{G_2\})=0.20,\; P(\{D_1\})=0.10,\; P(\{D_2\})=0.20.$$

---

### Part 2: $\sigma$-algebra generated by $X$

Since $X(G_1)=X(G_2)=0$ and $X(D_1)=X(D_2)=1$, the preimages of $X$ are:
$$X^{-1}(\{0\}) = \{G_1,G_2\}, \quad X^{-1}(\{1\}) = \{D_1,D_2\}, \quad X^{-1}(\emptyset)=\emptyset, \quad X^{-1}(\mathbb{R})=\Omega.$$

Therefore:
$$\sigma(X) = \{\emptyset,\; \{G_1,G_2\},\; \{D_1,D_2\},\; \Omega\}.$$
This is the coarsest $\sigma$-algebra that makes $X$ measurable — it only distinguishes "good" from "defective."

---

### Part 3: $\sigma$-algebra generated by $Y$

Since $Y$ takes four distinct values (1, 2, 3, 4) on four distinct outcomes, every singleton is a preimage:
$$Y^{-1}(\{1\})=\{G_1\},\; Y^{-1}(\{2\})=\{G_2\},\; Y^{-1}(\{3\})=\{D_1\},\; Y^{-1}(\{4\})=\{D_2\}.$$

Therefore $\sigma(Y)$ is generated by all singletons:
$$\sigma(Y) = \mathfrak{P}(\Omega) = \text{all 16 subsets of }\Omega.$$

---

### Part 4: $\sigma(X) \subseteq \sigma(Y)$

Since $\sigma(Y) = \mathfrak{P}(\Omega)$ contains every subset of $\Omega$, it certainly contains $\emptyset$, $\{G_1,G_2\}$, $\{D_1,D_2\}$, and $\Omega$ — i.e., every element of $\sigma(X)$. Hence $\sigma(X) \subseteq \sigma(Y)$.

**Why $Y$ contains more information than $X$:** $\sigma(Y) = \mathfrak{P}(\Omega)$ is the **finest** possible $\sigma$-algebra on $\Omega$ — it distinguishes every individual outcome. By contrast, $\sigma(X)$ only has four sets; it cannot distinguish $G_1$ from $G_2$, or $D_1$ from $D_2$. Knowing the value of $Y$ tells you exactly which machine produced the item, while knowing $X$ only tells you good vs. defective.

---

### Part 5: Marginal distribution of $X$

| $X$ | 0 (good) | 1 (defective) |
|-----|----------|---------------|
| $P(X=\cdot)$ | $0.50+0.20=0.70$ | $0.10+0.20=0.30$ |

---

### Part 6: Marginal distribution of $Y$

| $Y$ | 1 ($G_1$) | 2 ($G_2$) | 3 ($D_1$) | 4 ($D_2$) |
|-----|-----------|-----------|-----------|----------|
| $P(Y=\cdot)$ | 0.50 | 0.20 | 0.10 | 0.20 |

---

### Part 7: Joint probabilities

Because $X$ is a function of $Y$ ($X = \mathbf{1}_{\{Y \in \{3,4\}\}}$), the joint distribution is degenerate — each joint atom $(X=x, Y=y)$ coincides with a single atomic event:

$$P(X=0, Y=1) = P(\{G_1\}) = 0.50,$$
$$P(X=0, Y=2) = P(\{G_2\}) = 0.20,$$
$$P(X=1, Y=3) = P(\{D_1\}) = 0.10,$$
$$P(X=1, Y=4) = P(\{D_2\}) = 0.20.$$

All other joint probabilities (e.g., $P(X=0,Y=3)$) are 0.

---

### Part 8: Conditional probabilities

$$P(X=1 \mid Y=3) = \frac{P(X=1,Y=3)}{P(Y=3)} = \frac{0.10}{0.10} = 1.$$
$$P(X=1 \mid Y=4) = \frac{P(X=1,Y=4)}{P(Y=4)} = \frac{0.20}{0.20} = 1.$$
$$P(Y=3 \mid X=1) = \frac{P(X=1,Y=3)}{P(X=1)} = \frac{0.10}{0.30} = \frac{1}{3}.$$
$$P(Y=4 \mid X=1) = \frac{P(X=1,Y=4)}{P(X=1)} = \frac{0.20}{0.30} = \frac{2}{3}.$$

---

### Part 9: Conditioning on $X$ vs. conditioning on $Y$

- **Conditioning on $Y$:** Since $Y$ determines $X$ completely, knowing $Y=3$ tells us with certainty that $X=1$ (defective). There is no residual uncertainty about $X$ once $Y$ is known. The conditional distribution of $X$ given $Y$ is a point mass.

- **Conditioning on $X$:** Knowing $X=1$ tells us the item is defective, but does **not** tell us which machine produced it. The conditional distribution of $Y$ given $X=1$ is non-trivial: $P(Y=3 \mid X=1)=1/3$ and $P(Y=4 \mid X=1)=2/3$. Machine 2 is twice as likely to be responsible for a defect, given equal marginal defect rates.

---

### Part 10: Information in $\sigma(X)$ vs. $\sigma(Y)$

- **Information in $\sigma(X)$ but not $\sigma(Y)$:** None. Every set in $\sigma(X)$ is also in $\sigma(Y)$ (since $\sigma(X) \subsetneq \sigma(Y)$).

- **Information in $\sigma(Y)$ but not $\sigma(X)$:** All 12 extra subsets in $\mathfrak{P}(\Omega) \setminus \sigma(X)$, e.g., $\{G_1\}$, $\{G_2\}$, $\{D_1\}$, $\{G_1,D_1\}$, etc. In practical terms, $\sigma(Y)$ tells you **which machine** produced the item, while $\sigma(X)$ does not. This machine-identity information is present in $\sigma(Y)$ but is invisible to $\sigma(X)$.

---

### Part 12: Shannon Entropies

Using $H = -\sum_i p_i \log_2 p_i$:

**Marginal entropy of $X$:**
$$H(X) = -0.70\log_2(0.70) - 0.30\log_2(0.30) \approx 0.8813 \text{ bits.}$$

**Marginal entropy of $Y$:**
$$H(Y) = -0.50\log_2(0.50) - 0.20\log_2(0.20) - 0.10\log_2(0.10) - 0.20\log_2(0.20) \approx 1.7610 \text{ bits.}$$

**Joint entropy $H(X,Y)$:** Since $X$ is determined by $Y$, knowing $Y$ determines both, so:
$$H(X,Y) = H(Y) \approx 1.7610 \text{ bits.}$$

**Conditional entropy $H(X \mid Y)$:** Since $Y$ determines $X$:
$$H(X \mid Y) = H(X,Y) - H(Y) = 0 \text{ bits.}$$

**Conditional entropy $H(Y \mid X)$:**
$$H(Y \mid X) = H(X,Y) - H(X) = H(Y) - H(X) \approx 1.7610 - 0.8813 = 0.8797 \text{ bits.}$$

**Mutual information $I(X;Y)$:**
$$I(X;Y) = H(X) - H(X \mid Y) = H(X) = 0.8813 \text{ bits.}$$

The mutual information equals $H(X)$ because $Y$ tells you everything about $X$. The residual uncertainty $H(Y\mid X) \approx 0.88$ bits is the machine-identity information that $X$ cannot recover.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Probabilities
outcomes = ['G1', 'G2', 'D1', 'D2']
probs = np.array([0.50, 0.20, 0.10, 0.20])
X_vals = np.array([0, 0, 1, 1])   # 0=good, 1=defective
Y_vals = np.array([1, 2, 3, 4])

P_X = np.array([probs[X_vals==0].sum(), probs[X_vals==1].sum()])
H = lambda p: -np.sum(p[p>0] * np.log2(p[p>0]))

H_X = H(P_X)
H_Y = H(probs)
H_XY = H_Y  # X determined by Y
H_X_given_Y = 0.0
H_Y_given_X = H_XY - H_X
I_XY = H_X

print("Shannon Entropy Analysis")
print(f"H(X) = {H_X:.4f} bits")
print(f"H(Y) = {H_Y:.4f} bits")
print(f"H(X,Y) = H(Y) = {H_XY:.4f} bits  [since X = f(Y)]")
print(f"H(X|Y) = {H_X_given_Y:.4f} bits  [Y determines X completely]")
print(f"H(Y|X) = {H_Y_given_X:.4f} bits  [machine identity not in X]")
print(f"I(X;Y) = {I_XY:.4f} bits  [= H(X)]")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Probability distributions
ax = axes[0]
x_pos = np.arange(4)
bars = ax.bar(x_pos, probs, color=['#90EE90','#90EE90','#FF9999','#FF9999'],
              edgecolor='black', lw=1.2)
ax.set_xticks(x_pos)
ax.set_xticklabels(['G₁\n(Y=1)', 'G₂\n(Y=2)', 'D₁\n(Y=3)', 'D₂\n(Y=4)'], fontsize=11)
ax.set_ylabel('Probability', fontsize=11)
ax.set_title('Probability Distribution of Y\n(and underlying Ω)', fontsize=11)
for bar, p in zip(bars, probs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{p:.2f}',
            ha='center', fontsize=10)
ax.set_ylim(0, 0.65)
ax.legend(handles=[mpatches.Patch(color='#90EE90',label='Good (X=0)'),
                   mpatches.Patch(color='#FF9999',label='Defective (X=1)')], fontsize=9)

# Panel 2: sigma-algebra comparison
ax2 = axes[1]
ax2.axis('off')
ax2.set_xlim(0, 6); ax2.set_ylim(0, 6)
ax2.set_title('σ-Algebra Comparison\nσ(X) ⊆ σ(Y) = P(Ω)', fontsize=11)
# Draw nested boxes
outer = mpatches.FancyBboxPatch((0.2, 0.2), 5.6, 5.6, boxstyle='round,pad=0.1',
                                 facecolor='#E8F4FD', edgecolor='navy', lw=2)
inner = mpatches.FancyBboxPatch((1.2, 1.2), 3.6, 3.6, boxstyle='round,pad=0.1',
                                 facecolor='#FFF3CD', edgecolor='darkorange', lw=2)
ax2.add_patch(outer)
ax2.add_patch(inner)
ax2.text(3, 5.5, 'σ(Y) = P(Ω) [all 16 subsets]', ha='center', va='center', fontsize=9, color='navy')
ax2.text(3, 3, 'σ(X) = {∅, {G₁,G₂}, {D₁,D₂}, Ω}\n4 sets', ha='center', va='center',
         fontsize=9, color='darkorange', fontweight='bold')
ax2.text(0.5, 0.5, 'Extra in σ(Y): {G₁},{G₂},{D₁},{D₂},\n{G₁,D₁}, ... (12 more sets)',
         ha='left', va='bottom', fontsize=8, color='steelblue')

# Panel 3: Entropy diagram
ax3 = axes[2]
entropies = {'H(X)': H_X, 'H(Y)': H_Y, 'H(X|Y)': H_X_given_Y,
             'H(Y|X)': H_Y_given_X, 'I(X;Y)': I_XY}
names = list(entropies.keys())
vals = list(entropies.values())
colors_e = ['steelblue','green','red','orange','purple']
bars3 = ax3.barh(names, vals, color=colors_e, edgecolor='black', lw=0.8)
for bar, v in zip(bars3, vals):
    ax3.text(v + 0.02, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=10)
ax3.set_xlabel('Bits', fontsize=11)
ax3.set_title('Shannon Entropy Summary', fontsize=11)
ax3.set_xlim(0, 2.3)

plt.suptitle('Q9: Information in Discrete Random Variables X and Y', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Q10) Information Available in Continuous Random Variables

## Answer to Q10

**Setup:** $T \sim N(0,1)$, $P \sim N(0,4)$ independent. $X(t,p)=t$, $Y(t,p)=(t,p)$.

### Part 1: Probability Space

$$\Omega = \mathbb{R}^2, \quad \mathfrak{F} = \mathfrak{B}(\mathbb{R}^2),$$
with probability measure $\mathbb{P}$ induced by the joint density
$$f_{T,P}(t,p) = \frac{1}{4\pi}\exp\!\left(-\frac{t^2}{2} - \frac{p^2}{8}\right), \quad (t,p)\in\mathbb{R}^2.$$
This factors as $f_T(t)\cdot f_P(p)$ with $f_T(t)=\frac{1}{\sqrt{2\pi}}e^{-t^2/2}$ and $f_P(p)=\frac{1}{2\sqrt{2\pi}}e^{-p^2/8}$, confirming independence.

---

### Part 2: $\sigma(X)$

$X(t,p)=t$ depends only on $t$. Its preimages are horizontal "slabs":
$$X^{-1}(B) = B \times \mathbb{R}, \quad B \in \mathfrak{B}(\mathbb{R}).$$
Therefore:
$$\sigma(X) = \{B \times \mathbb{R} : B \in \mathfrak{B}(\mathbb{R})\}.$$
These are the sets that depend only on the first coordinate $t$; they are constant in the $p$-direction.

---

### Part 3: $\sigma(Y)$

$Y(t,p)=(t,p)$ is the identity map on $\mathbb{R}^2$. Its preimages are:
$$Y^{-1}(C) = C, \quad C \in \mathfrak{B}(\mathbb{R}^2).$$
Therefore:
$$\sigma(Y) = \mathfrak{B}(\mathbb{R}^2).$$

---

### Part 4: $\sigma(X) \subseteq \sigma(Y)$

Every set in $\sigma(X)$ has the form $B \times \mathbb{R}$ for some $B \in \mathfrak{B}(\mathbb{R})$. Since $B \times \mathbb{R}$ is a Borel subset of $\mathbb{R}^2$, it belongs to $\mathfrak{B}(\mathbb{R}^2) = \sigma(Y)$. Hence $\sigma(X) \subseteq \sigma(Y)$.

**$Y$ contains more information than $X$:** $\sigma(X)$ consists only of "vertical strips" (sets that are infinite in the $p$-direction), while $\sigma(Y) = \mathfrak{B}(\mathbb{R}^2)$ contains all Borel sets in the plane — including horizontal strips, rectangles, and arbitrary measurable regions. Knowing $Y=(t,p)$ reveals both the temperature and pressure deviations, whereas knowing $X=t$ reveals only the temperature.

---

### Part 5: Marginal density of $X$

Integrate out $p$:
$$f_X(x) = \int_{-\infty}^{\infty} f_{T,P}(x,p)\,dp = \frac{1}{4\pi}e^{-x^2/2}\int_{-\infty}^{\infty}e^{-p^2/8}\,dp = \frac{1}{4\pi}e^{-x^2/2}\cdot 2\sqrt{2\pi} = \frac{1}{\sqrt{2\pi}}e^{-x^2/2}.$$
$$\boxed{f_X(x) = \frac{1}{\sqrt{2\pi}}e^{-x^2/2}, \quad x\in\mathbb{R}.}$$
So $X \sim N(0,1)$ as expected.

---

### Part 6: Marginal density of $Y$

Since $Y=(T,P)$ is itself the joint state, its density is the joint density:
$$f_Y(t,p) = f_{T,P}(t,p) = \frac{1}{4\pi}\exp\!\left(-\frac{t^2}{2}-\frac{p^2}{8}\right), \quad (t,p)\in\mathbb{R}^2.$$
This is the bivariate normal $N(\mathbf{0},\,\text{diag}(1,4))$.

---

### Part 7: Probabilities

$$P(X \le 0) = P(T \le 0) = \Phi(0) = \frac{1}{2} = 0.5.$$

$$P(X > 1) = P(T > 1) = 1 - \Phi(1) \approx 0.1587.$$

$$P(Y \in (-\infty,0] \times \mathbb{R}) = P(T \le 0) = \frac{1}{2} = 0.5.$$

$$P(Y \in [-1,1]\times[-2,2]) = P(-1\le T \le 1)\cdot P(-2\le P \le 2)$$
$$= [\Phi(1)-\Phi(-1)] \cdot \left[\Phi\!\left(\frac{2}{2}\right) - \Phi\!\left(\frac{-2}{2}\right)\right] = [2\Phi(1)-1]^2 \approx 0.6827^2 \approx 0.4661.$$

---

### Part 8: Conditional distribution of $X$ given $Y=(t,p)$

Since $X(t,p) = t$, conditioning on $Y=(t,p)$ fixes the outcome completely. The conditional distribution of $X$ given $Y=(t,p)$ is the **point mass** (Dirac delta) at $t$:
$$\mathcal{L}(X \mid Y=(t,p)) = \delta_t.$$
In other words, $X = t$ with probability 1 given $Y=(t,p)$.

---

### Part 9: Conditional distribution of $Y$ given $X=t$

Given $X=T=t$, we know the first coordinate exactly but $P$ is independent of $T$:
$$\mathcal{L}(Y \mid X=t) = \mathcal{L}((t,P)) = \delta_t \otimes N(0,4).$$
Concretely: the conditional density of $Y=(t',p)$ given $X=t$ is $\delta(t'-t)\cdot f_P(p)$. The pressure component $P$ retains its full $N(0,4)$ distribution.

---

### Part 10: What conditioning reveals

- **Conditioning on $X=t$:** We know the temperature deviation is $t$. Since $T$ and $P$ are independent, pressure is still unknown and distributed as $N(0,4)$. $\sigma(X)$ carries only temperature information.

- **Conditioning on $Y=(t,p)$:** Both temperature and pressure deviations are fully known. No uncertainty remains about either coordinate. $\sigma(Y)$ carries the full bivariate state of the system.

---

### Part 11: $X = \pi_1 \circ Y$

Define the projection $\pi_1 : \mathbb{R}^2 \to \mathbb{R}$ by $\pi_1(t,p) = t$. Then
$$X(t,p) = t = \pi_1(t,p) = (\pi_1 \circ Y)(t,p).$$
Since $Y: \Omega \to \mathbb{R}^2$ and $\pi_1: \mathbb{R}^2 \to \mathbb{R}$ is a continuous (hence Borel-measurable) function, the composition $X = \pi_1 \circ Y$ is measurable with respect to $\sigma(Y)$. That is, for every $B \in \mathfrak{B}(\mathbb{R})$,
$$X^{-1}(B) = (\pi_1 \circ Y)^{-1}(B) = Y^{-1}(\pi_1^{-1}(B)) = Y^{-1}(B \times \mathbb{R}) \in \sigma(Y).$$
This confirms $\sigma(X) \subseteq \sigma(Y)$: $X$ is a measurable function of $Y$, so the information in $X$ is a subset of the information in $Y$.

---

### Part 13: Differential entropy of $X$

For $X \sim N(0,1)$, the differential entropy is:
$$h(X) = -\int_{-\infty}^{\infty} f_X(x)\ln f_X(x)\,dx = \frac{1}{2}\ln(2\pi e) \approx 1.4189 \text{ nats.}$$

---

### Part 14: Differential entropy of $Y$

Since $T$ and $P$ are independent:
$$h(Y) = h(T,P) = h(T) + h(P) = \frac{1}{2}\ln(2\pi e) + \frac{1}{2}\ln(2\pi e \cdot 4) = \frac{1}{2}\ln(2\pi e) + \frac{1}{2}\ln(8\pi e).$$
$$h(Y) = \frac{1}{2}\ln(2\pi e) + \frac{1}{2}\ln(2\pi e) + \frac{1}{2}\ln 4 = \ln(2\pi e) + \ln 2 \approx 3.5310 \text{ nats.}$$

---

### Part 15: Conditional differential entropy $h(Y \mid X)$

Since $T$ and $P$ are independent, knowing $T=t$ gives no information about $P$:
$$h(Y \mid X) = h((T,P) \mid T) = h(P \mid T) + h(T \mid T) = h(P) + 0 = \frac{1}{2}\ln(8\pi e) \approx 2.1121 \text{ nats.}$$

**Physical interpretation:** After observing the temperature deviation $T$, the remaining uncertainty in the full system state $(T,P)$ is entirely due to the unknown pressure deviation $P$, which has entropy $h(P) = \frac{1}{2}\ln(8\pi e)$ nats. The pressure contains more entropy than the temperature because $\text{Var}(P)=4 > \text{Var}(T)=1$.

---

### Part 16: Conditional entropy $h(X \mid Y)$ — continuous vs. discrete

Since $X = \pi_1 \circ Y$, knowing $Y=(t,p)$ determines $X=t$ with **certainty**. One might expect $h(X \mid Y) = 0$ by analogy with the discrete case (where $H(X \mid Y) = 0$ whenever $X$ is a function of $Y$).

However, in the **continuous case**, conditioning on a continuous random variable $Y$ is defined via regular conditional distributions, and:
$$h(X \mid Y) = h(X,Y) - h(Y) = h(Y) - h(Y) = 0.$$

So indeed $h(X \mid Y) = 0$ in this model, in agreement with the discrete analogy — because $X$ is an exact function of $Y$, no entropy remains in $X$ once $Y$ is known.

**Why the continuous case can differ in general:** Unlike the discrete case where $H(X \mid Y) \ge 0$ always, the **differential entropy** $h(X \mid Y)$ can be **negative** (differential entropy is not a true measure of uncertainty — it lacks the non-negativity property of Shannon entropy). Moreover, for arbitrary jointly continuous $(X,Y)$, even if $X = g(Y)$ for a non-injective $g$, we still get $h(X \mid Y) = 0$. But if we condition on $Y$ and $X$ is not determined by $Y$, we can get $h(X \mid Y) < 0$ in principle. The key structural difference: in the discrete case, $H(X \mid Y)=0 \iff X$ is a function of $Y$; in the continuous case, $h(X \mid Y)=0$ whenever $X$ is measurable w.r.t. $\sigma(Y)$, but $h(X \mid Y)$ can in general be any real number.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import math

# Numerical verification
print("=== Q10 Numerical Verification ===")
print(f"P(X ≤ 0) = {stats.norm.cdf(0, 0, 1):.4f}  (exact: 0.5)")
print(f"P(X > 1) = {1 - stats.norm.cdf(1, 0, 1):.4f}  (exact ≈ 0.1587)")
print(f"P(Y ∈ (-∞,0]×ℝ) = {stats.norm.cdf(0, 0, 1):.4f}")
p_T = stats.norm.cdf(1,0,1) - stats.norm.cdf(-1,0,1)
p_P = stats.norm.cdf(2,0,2) - stats.norm.cdf(-2,0,2)
print(f"P(Y ∈ [-1,1]×[-2,2]) = {p_T:.4f} × {p_P:.4f} = {p_T*p_P:.4f}")

h_X = 0.5 * math.log(2*math.pi*math.e)
h_P_var = 0.5 * math.log(2*math.pi*math.e*4)
h_Y = h_X + h_P_var
h_Y_given_X = h_P_var
print(f"\nh(X) = {h_X:.4f} nats")
print(f"h(Y) = {h_Y:.4f} nats")
print(f"h(Y|X) = h(P) = {h_Y_given_X:.4f} nats")
print(f"h(X|Y) = 0.0000 nats  [X determined by Y]")

# Visualizations
fig = plt.figure(figsize=(15, 10))

# 1. Joint density heatmap
ax1 = fig.add_subplot(2, 3, 1)
t_range = np.linspace(-4, 4, 200)
p_range = np.linspace(-8, 8, 200)
T_grid, P_grid = np.meshgrid(t_range, p_range)
joint_density = (1/(4*np.pi)) * np.exp(-T_grid**2/2 - P_grid**2/8)
cp = ax1.contourf(T_grid, P_grid, joint_density, levels=20, cmap='viridis')
plt.colorbar(cp, ax=ax1)
ax1.set_xlabel('t (Temperature)', fontsize=10)
ax1.set_ylabel('p (Pressure)', fontsize=10)
ax1.set_title('Joint Density of Y=(T,P)', fontsize=11)
ax1.axhline(0, color='white', lw=0.5, alpha=0.5)
ax1.axvline(0, color='white', lw=0.5, alpha=0.5)

# 2. Marginal density of X
ax2 = fig.add_subplot(2, 3, 2)
x_range = np.linspace(-4, 4, 300)
f_X = stats.norm.pdf(x_range, 0, 1)
ax2.plot(x_range, f_X, 'b-', lw=2.5)
ax2.fill_between(x_range[x_range<=0], f_X[x_range<=0], alpha=0.3, color='green', label='P(X≤0)=0.5')
ax2.fill_between(x_range[x_range>=1], f_X[x_range>=1], alpha=0.3, color='red', label='P(X>1)≈0.159')
ax2.set_xlabel('x', fontsize=10)
ax2.set_ylabel('Density', fontsize=10)
ax2.set_title('Marginal Density of X ~ N(0,1)', fontsize=11)
ax2.legend(fontsize=9)

# 3. Sigma-algebra diagram
ax3 = fig.add_subplot(2, 3, 3)
ax3.axis('off')
ax3.set_xlim(-5, 5); ax3.set_ylim(-5, 5)
ax3.set_title('σ(X) vs σ(Y) in ℝ²', fontsize=11)
# Draw vertical strips for sigma(X)
for x0 in np.linspace(-4, 2, 4):
    rect = plt.Rectangle((x0, -5), 0.8, 10, alpha=0.15, color='blue')
    ax3.add_patch(rect)
ax3.axvline(-1, color='blue', lw=2, linestyle='--', label='σ(X): vertical strips (B×ℝ)')
# Draw a Borel set (circle) for sigma(Y)
circle = plt.Circle((1, 1), 1.5, fill=True, facecolor='red', alpha=0.2,
                     edgecolor='red', lw=2, linestyle='-', label='σ(Y): any Borel set')
ax3.add_patch(circle)
ax3.set_xlim(-4, 4); ax3.set_ylim(-4, 4)
ax3.set_aspect('equal')
ax3.legend(fontsize=8, loc='upper left')
ax3.set_xlabel('t (temperature)', fontsize=9)
ax3.set_ylabel('p (pressure)', fontsize=9)
ax3.text(0, -3.5, 'Blue strips ∈ σ(X) ⊂ σ(Y)\nRed circle ∈ σ(Y) but ∉ σ(X)',
         ha='center', fontsize=8, color='purple')

# 4. Entropy comparison bar chart
ax4 = fig.add_subplot(2, 3, 4)
labels = ['h(X)', 'h(P)', 'h(Y)=h(T)+h(P)', 'h(Y|X)=h(P)', 'h(X|Y)']
values = [h_X, h_P_var, h_Y, h_Y_given_X, 0]
colors_h = ['steelblue','darkorange','green','darkorange','red']
ax4.bar(range(len(labels)), values, color=colors_h, edgecolor='black')
ax4.set_xticks(range(len(labels)))
ax4.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)
ax4.set_ylabel('Nats', fontsize=10)
ax4.set_title('Differential Entropy Summary', fontsize=11)
for i, v in enumerate(values):
    ax4.text(i, v+0.05, f'{v:.3f}', ha='center', fontsize=9)
ax4.set_ylim(0, 4.2)

# 5. Conditional distributions illustration
ax5 = fig.add_subplot(2, 3, 5)
t_fixed = 1.0
p_range2 = np.linspace(-8, 8, 300)
f_P_given_T = stats.norm.pdf(p_range2, 0, 2)
ax5.plot(p_range2, f_P_given_T, 'r-', lw=2.5, label=f'f(P|T={t_fixed}) = N(0,4)')
ax5.axvline(t_fixed, color='blue', lw=2.5, linestyle='--', label=f'f(X|Y=({t_fixed},p)) = δ_{{t={t_fixed}}}')
ax5.set_xlabel('Value', fontsize=10)
ax5.set_ylabel('Density', fontsize=10)
ax5.set_title(f'Conditional Distributions\n(given T={t_fixed})', fontsize=11)
ax5.legend(fontsize=8)

# 6. Monte Carlo verification
ax6 = fig.add_subplot(2, 3, 6)
np.random.seed(42)
T_samples = np.random.normal(0, 1, 2000)
P_samples = np.random.normal(0, 2, 2000)
ax6.scatter(T_samples, P_samples, alpha=0.1, s=5, color='steelblue')
# Highlight Y in [-1,1]x[-2,2]
mask = (np.abs(T_samples) <= 1) & (np.abs(P_samples) <= 2)
ax6.scatter(T_samples[mask], P_samples[mask], alpha=0.3, s=5, color='red', label='In [-1,1]×[-2,2]')
ax6.add_patch(plt.Rectangle((-1,-2), 2, 4, fill=False, edgecolor='red', lw=2))
ax6.set_xlabel('T (temperature)', fontsize=10)
ax6.set_ylabel('P (pressure)', fontsize=10)
ax6.set_title(f'Monte Carlo: P(Y∈[-1,1]×[-2,2])\n≈ {mask.mean():.3f} (exact: {p_T*p_P:.3f})', fontsize=10)
ax6.legend(fontsize=8)

plt.suptitle('Q10: Information in Continuous Random Variables T and P', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()